# Colab 부트스트랩 (VS Code + Google Colab 확장)

VS Code에서 이 노트북을 **Colab GPU 서버**에 연결해 실험 환경을 준비한다. Google Drive에 업로드할 필요가 없다.

1. GitHub 저장소를 서버의 `/content/Inthon_team01`에 clone하거나 pull한다.
2. 채점 환경과 같은 버전(Python 3.12.3, torch 2.8.0+cu128, transformers 4.57.6 등)의 가상환경을 `/content/inthon-py312`에 만든다. 버전 목록은 운영진 `starterkit/starterkit_colab.ipynb`와 같다.
3. Hugging Face에서 Train을 받아 `/content/Inthon_team01/data/Train`에 푼다(SHA256 검증). `starterkit/config.json`의 기본 경로와 같다.
4. baseline 인코더(`facebook/dinov2-base`, `sentence-transformers/all-mpnet-base-v2`)를 `starterkit/models/`에 받는다. SigLIP과 SmolVLM은 선택이다.
5. GPU, 패키지 버전, bitsandbytes 4bit 연산, starterkit 단위 테스트를 확인한다.

**연결 방법:** 오른쪽 위 `Select Kernel` → `Colab` → `New Colab Server` → **T4 GPU** 선택 → Google 로그인. 그다음 위에서부터 차례로 실행한다.

**주의**
- 서버의 `/content`는 서버를 지우거나 세션이 끊기면 사라진다. 남길 결과는 Drive(아래 `MOUNT_DRIVE`)나 git에 저장한다.
- VS Code 확장에서는 `google.colab.userdata`(Secrets)와 `files.download()`/`files.upload()`가 동작하지 않는다. 토큰이 필요하면 `getpass`로 입력하고, 파일은 Colab 탐색기의 `Download...` 또는 `Upload to Colab`을 쓴다.
- 일반 셀의 `import torch`는 Colab 기본 환경이다. 대회 버전으로 실행하려면 아래 `run([PYTHON, ...])`을 쓴다.

## 0. 설정과 GPU 확인

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/TTKKWAN/Inthon_team01.git"
BRANCH = "Hwooma"                          # 실행할 브랜치
REPO_DIR = Path("/content/Inthon_team01")
ENV_DIR = Path("/content/inthon-py312")    # 채점 환경과 같은 버전의 가상환경
TOOL_DIR = Path("/content/inthon-tools")   # uv 설치 위치(Colab 기본 환경을 건드리지 않음)
HF_HOME = Path("/content/hf-cache")
DOWNLOAD_EXTRA_MODELS = False              # True면 SigLIP, SmolVLM도 /content/models 에 받는다
MOUNT_DRIVE = False                        # True면 /content/drive 에 Google Drive를 연결한다

if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU 서버가 아닙니다. Select Kernel > Colab > New Colab Server 에서 T4 GPU를 고르세요.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader"], check=True)

ENV = os.environ.copy()
for name in ("PYTHONPATH", "PYTHONHOME", "HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    ENV.pop(name, None)
ENV.update({"HF_HOME": str(HF_HOME), "TOKENIZERS_PARALLELISM": "false",
            "PYTHONUNBUFFERED": "1", "PYTHONNOUSERSITE": "1"})


def run(args, cwd=None, extra_env=None):
    """명령을 실행하며 출력을 셀에 흘려 보낸다. 실패하면 예외를 낸다."""
    command = [str(arg) for arg in args]
    print("$", " ".join(command), flush=True)
    with subprocess.Popen(command, cwd=cwd, env={**ENV, **(extra_env or {})},
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

## 1. 저장소 clone / pull

로컬에서 코드를 고쳐 `BRANCH`에 push했다면 이 셀만 다시 실행해 최신 코드로 맞춘다.

In [ ]:
if (REPO_DIR / ".git").is_dir():
    run(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
run(["git", "log", "-1", "--format=%h %an %ad %s", "--date=iso"], cwd=REPO_DIR)

## 2. 채점 환경과 같은 가상환경

운영진 노트북과 같은 버전 목록이다(채점 환경 기준일 2026-10-31). 처음에는 수 분 걸리고, 다시 실행하면 이미 설치된 패키지는 건너뛴다.
런타임 재시작은 필요 없다. 대회 버전으로 실행할 코드는 모두 `PYTHON`으로 실행한다.

In [ ]:
PYTHON_VERSION = "3.12.3"
TORCH_PINS = ["torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128"]
PACKAGE_PINS = [
    "transformers==4.57.6", "numpy==2.5.4", "Pillow==12.3.0", "safetensors==0.8.0",
    "huggingface-hub==0.36.2", "tokenizers==0.22.2",
    "sentence-transformers==5.4.0", "einops==0.8.2", "timm==1.0.30",
    "scipy==1.18.1", "scikit-learn==1.9.1", "opencv-python-headless==4.14.0.94",
    "peft==0.18.1", "accelerate==1.15.0", "bitsandbytes==0.50.2",
    "cloudpickle==3.1.2", "joblib==1.6.0", "narwhals==2.28.0", "threadpoolctl==3.7.0",
]
DEV_PINS = ["pytest==9.2.0"]  # 단위 테스트용. 제출 코드에서는 쓰지 않는다.

UV = TOOL_DIR / "bin/uv"
if not UV.is_file():
    run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
         "--target", TOOL_DIR, "uv==0.13.4"])
PYTHON = ENV_DIR / "bin/python"
if not PYTHON.is_file():
    run([UV, "python", "install", PYTHON_VERSION])
    run([UV, "venv", "--python", PYTHON_VERSION, "--seed", ENV_DIR])
run([PYTHON, "-c", "import sys; assert sys.version_info[:3] == (3, 12, 3), sys.version"])
run([PYTHON, "-m", "pip", "install", "--quiet", "--no-cache-dir",
     "--index-url", "https://download.pytorch.org/whl/cu128", *TORCH_PINS])
run([PYTHON, "-m", "pip", "install", "--quiet", "--no-cache-dir", *PACKAGE_PINS, *DEV_PINS])
run([PYTHON, "-m", "pip", "check"])
print("가상환경 준비 완료:", PYTHON)

## 3. Train 데이터

`scripts/download_data.py`로 받는다(약 4GB, SHA256 검증 후 압축 해제). 이미 있으면 건너뛴다.

In [ ]:
run([PYTHON, REPO_DIR / "scripts/download_data.py"], cwd=REPO_DIR)
TRAIN_DIR = REPO_DIR / "data/Train"
episodes = [p for p in TRAIN_DIR.iterdir() if p.is_dir() and p.name.startswith("episode_")]
print("episodes:", len(episodes))
assert len(episodes) == 4170, len(episodes)

## 4. 허용 모델 받기

baseline 인코더 두 개는 `starterkit/download_models.py`로 `starterkit/models/`에 받는다.
`DOWNLOAD_EXTRA_MODELS = True`이면 나머지 허용 모델 두 개도 `/content/models/`에 받는다.
SigLIP 텍스트 토크나이저는 sentencepiece가 필요해 채점 환경에서 `AutoTokenizer`로는 로드되지 않는다. 쓰기 전에 `tokenizer.json` 기반 fast 토크나이저와 결과가 같은지 확인해야 한다.

In [ ]:
run([PYTHON, "download_models.py"], cwd=REPO_DIR / "starterkit")

EXTRA_MODELS = r"""
import sys
from pathlib import Path
from huggingface_hub import snapshot_download
root = Path(sys.argv[1])
for name, repo in [("siglip", "google/siglip-base-patch16-224"),
                   ("smolvlm", "HuggingFaceTB/SmolVLM-256M-Instruct")]:
    path = snapshot_download(repo, local_dir=root / name,
                             allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"])
    print("ready:", repo, "->", path)
"""
if DOWNLOAD_EXTRA_MODELS:
    run([PYTHON, "-c", EXTRA_MODELS, "/content/models"])

## 5. 환경 점검

패키지 버전이 목록과 정확히 같은지, CUDA fp16 연산과 bitsandbytes nf4 연산이 되는지 확인하고 starterkit 단위 테스트를 돌린다.
Colab의 GPU와 드라이버는 채점 서버와 다를 수 있으니 시간과 수치가 완전히 같다는 뜻은 아니다.

In [ ]:
CHECK = r"""
import importlib.metadata as metadata, json, platform, sys
import torch
pins = json.loads(sys.argv[1])
wrong = {}
for pin in pins:
    name, expected = pin.split("==")
    found = metadata.version(name)
    if found != expected:
        wrong[name] = [found, expected]
assert not wrong, wrong
assert torch.cuda.is_available()
x = torch.randn(32, 64, device="cuda", dtype=torch.float16)
assert torch.isfinite(x @ x.T).all()
import bitsandbytes as bnb
layer = bnb.nn.Linear4bit(64, 16, bias=False, compute_dtype=torch.float16, quant_type="nf4").to("cuda")
with torch.inference_mode():
    assert torch.isfinite(layer(x)).all()
print(json.dumps({"python": platform.python_version(), "torch": torch.__version__,
                  "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
                  "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)},
                 indent=2))
"""
run([PYTHON, "-c", CHECK, json.dumps(TORCH_PINS + PACKAGE_PINS)])
run([PYTHON, "-m", "pytest", "-q", "tests"], cwd=REPO_DIR / "starterkit")

## 6. (선택) Google Drive 연결

서버가 사라져도 남겨야 할 결과(체크포인트, 제출 zip 등)를 저장할 때 쓴다. 명령 팔레트의 `Colab: Mount Google Drive to Server...`로도 연결할 수 있다.

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 다음 단계

- 스크립트는 대회 버전 환경으로 실행한다. 예: `run([PYTHON, "train.py", "--device", "cuda:0"], cwd=REPO_DIR / "starterkit")`
- 실험 결과(점수, 소요 시간, 설정)는 셀 출력으로 남기고 노트북을 저장한다. 로컬의 Claude Code가 저장된 출력을 읽어 실험 기록으로 정리하고 브랜치에 push한다.
- 가중치, 데이터, zip은 git에 올리지 않는다. Drive에 두거나 Colab 탐색기의 `Download...`로 받는다.